<a href="https://colab.research.google.com/github/gugomesx10/CP2-GEN_AI/blob/main/CP2_ia_generativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CHECKPOINT 02: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Gustavo Gomes Martins - RM555999
- Matheus de Mattos Vecchi - RM561716
- Nicholas Albuquerque Buzo	- RM561082
- Nicholas Camillo Canadas de Paula - RM561262

**Tema escolhido:** Suporte Técnico de IOT

---

Nesta avaliação o grupo vai aprofundar o assistente construído na atividade anterior, em 4 etapas (e 1 bônus):

1. Assistente com guardrails (restrições no `system`)
2. Comparação com temperatura
3. Chat com memória e troca de modelo
4. Interface web com Gradio e controle de temperatura
5. API com sessões independentes

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.  
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [16]:
!pip install google-genai gradio -q

In [15]:
from google import genai
from google.genai import types
from google.colab import userdata

In [17]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

GEMINI_API_KEY ok


In [22]:
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [19]:
# >>> PERSONALIZE: system prompt do tema do grupo.
# Um bom system diz: quem o assistente é, sobre o que fala,
# como responde (idioma, tamanho) e o que não deve fazer.
SYSTEM_PROMPT = """Você é um assistente especializado em suporte técnico de IoT.

Sua função é ajudar usuários a diagnosticar problemas com ESP32, Arduino, sensores, redes Wi-Fi e dispositivos conectados.

Forneça orientações técnicas claras, organizadas e adequadas para iniciantes.

Responda sempre em português, utilizando no máximo 5 frases.

Se a pergunta não estiver relacionada a IoT, informe educadamente que o assunto está fora da sua especialidade.

Não invente especificações técnicas e não recomende procedimentos elétricos perigosos."""

---
## Etapa 1: Assistente com guardrails

Na atividade anterior o grupo testou personalidades diferentes.
Aqui o foco é o **guardrail**: usar o `system` para restringir o assistente a responder **apenas** perguntas do tema.

A mesma pergunta será enviada com três versões do `system`:
- sem restrição
- com restrição de tema
- com restrição de tema e de formato

Também será testada uma pergunta **fora do escopo** para ver se o assistente respeita o guardrail.

In [20]:
def perguntar_gemini(pergunta, system, temperatura=0.7):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [23]:
# >>> PERSONALIZE: Etapa 1 - Assistente com Guardrails

versoes_system = {
    "Sem restrição": """Você é um assistente prestativo.
Responda sempre em português.""",

    "Com restrição de tema": """Você é um especialista em suporte técnico de IoT.
Responda apenas perguntas relacionadas a ESP32, Arduino,
sensores, redes Wi-Fi e dispositivos IoT.
Se a pergunta estiver fora desse tema, recuse educadamente.
Responda sempre em português.""",

    "Com restrição de tema e formato": """Você é um especialista em suporte técnico de IoT.
Responda apenas perguntas relacionadas a ESP32, Arduino,
sensores, redes Wi-Fi e dispositivos IoT.
Se a pergunta estiver fora desse tema, recuse educadamente.
Responda sempre em português, em no máximo 3 frases,
sem utilizar listas ou tópicos."""
}

# Perguntas para testar os guardrails
pergunta_dentro = "Por que meu ESP32 não consegue conectar ao Wi-Fi?"
pergunta_fora = "Qual é a capital da França?"

# Executa os testes utilizando apenas o Google Gemini
for nome, system in versoes_system.items():
    print(f"\n{'=' * 55}")
    print(f"  {nome}")
    print(f"{'=' * 55}")

    print(f"\n[Dentro do escopo] {pergunta_dentro}")
    print(perguntar_gemini(pergunta_dentro, system))

    print(f"\n[Fora do escopo] {pergunta_fora}")
    print(perguntar_gemini(pergunta_fora, system))

print("\nTeste de guardrails finalizado!")


  Sem restrição

[Dentro do escopo] Por que meu ESP32 não consegue conectar ao Wi-Fi?
Existem várias razões pelas quais o ESP32 pode falhar ao conectar-se ao Wi-Fi. Como o ESP32 é sensível a alguns detalhes de configuração e hardware, o problema geralmente se resume a uma das seguintes causas comuns. 

Aqui está um passo a passo para diagnosticar e resolver o problema:

### 1. SSID ou Senha Incorretos
Este é o erro mais comum. 
* **Verifique maiúsculas e minúsculas:** O ESP32 diferencia letras maiúsculas de minúsculas tanto no nome da rede (SSID) quanto na senha.
* **Redes ocultas:** Se o seu roteador oculta o SSID, você precisa configurar explicitamente o ESP32 para se conectar a redes ocultas (dependendo da biblioteca que estiver usando).

### 2. Problemas com a Frequência (2.4 GHz vs 5 GHz)
* **O ESP32 só suporta Wi-Fi de 2.4 GHz.** Ele **não** consegue se conectar a redes de 5 GHz.
* Se o seu roteador usa "Smart Connect" (mesmo nome para as redes de 2.4 GHz e 5 GHz), o ESP32 pode 

**Observações do grupo (Etapa 1):**

- O assistente sem restrição respondeu a pergunta fora do escopo?
- Com o guardrail de tema, ele recusou? A recusa foi educada?
- A restrição de formato mudou o estilo da resposta?

(responda aqui)

---
## Etapa 2: Comparação com temperatura

Na atividade anterior o grupo comparou Hugging Face e Gemini com o mesmo prompt.
Aqui o grupo compara o efeito da **temperatura** nos dois modelos.

- `temperature=0.0` — o modelo escolhe sempre o token mais provável
- `temperature=1.0` — o modelo amostra com mais aleatoriedade

A mesma pergunta é enviada com cada valor de temperatura para os dois modelos.

In [24]:
def perguntar_gemini(pergunta, system, temperatura=1.0):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [25]:
# Etapa 2 - Comparação de Temperaturas
# Tema: Suporte Técnico de IoT

perguntas = [
    "Quais são as principais causas de falha na leitura de um sensor DHT22 conectado ao ESP32?",
    "Como melhorar a segurança da comunicação Wi-Fi entre dispositivos IoT?"
]

temperaturas = [0.0, 1.0]

for pergunta in perguntas:
    print("\n" + "=" * 60)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 60)

    for temperatura in temperaturas:
        print(f"\n--- Gemini | Temperatura: {temperatura} ---")

        resposta = perguntar_gemini(
            pergunta,
            SYSTEM_PROMPT,
            temperatura=temperatura
        )

        print(resposta)
        print("-" * 60)

print("\nComparação de temperaturas finalizada!")


PERGUNTA: Quais são as principais causas de falha na leitura de um sensor DHT22 conectado ao ESP32?

--- Gemini | Temperatura: 0.0 ---
As principais causas de falha na leitura do sensor DHT22 com o ESP32 são problemas na fiação, alimentação instável e ausência de resistor de pull-up. Verifique se o pino de dados está conectado à porta GPIO correta configurada no seu código e se a alimentação de 3,3V está firme. O uso de cabos muito longos também pode causar perda de sinal na comunicação de um único fio. Por fim, certifique-se de respeitar o intervalo mínimo de dois segundos entre as leituras exigido pelo sensor.
------------------------------------------------------------

--- Gemini | Temperatura: 1.0 ---
As falhas na leitura do sensor DHT22 no ESP32 geralmente ocorrem por problemas na conexão física, como cabos invertidos ou mal conectados. Outra causa comum é a ausência de um resistor de pull-up (geralmente de 4,7kΩ a 10kΩ) entre o pino de dados e a alimentação de 3,3V. Problemas d

**Observações do grupo (Etapa 2):**

- Com `temperature=0.0`, as respostas dos dois modelos ficaram parecidas entre si?
- Com `temperature=1.0`, as respostas ficaram mais variadas?
- Qual combinação de modelo e temperatura vocês consideram melhor para o tema?

(responda aqui)

---
## Etapa 3: Chat com memória e troca de modelo

Na atividade anterior o chat usava apenas o Hugging Face.
Aqui o grupo pode trocar o modelo durante a conversa, mantendo o histórico.

Comandos:
- `sair` — encerra o chat
- `limpar` — apaga o histórico
- `historico` — mostra quantas mensagens estão guardadas
- `modelo hf` — troca para Hugging Face
- `modelo gemini` — troca para Gemini

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", troque o modelo e pergunte de novo.

In [27]:
# Etapa 3 - Chat com memória usando Google Gemini

def responder_chat(historico):
    conteudos = []

    for msg in historico:
        if msg["role"] == "system":
            continue

        papel = "model" if msg["role"] == "assistant" else "user"

        conteudos.append({
            "role": papel,
            "parts": [{"text": msg["content"]}]
        })

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=0.7
        ),
    )

    return resposta.text

In [28]:
# Etapa 3 - Chat com memória usando Google Gemini
# Tema: Suporte Técnico de IoT

historico = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

print("Chat iniciado com Google Gemini!")
print("Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ").strip()

    if entrada.lower() == "sair":
        print("\nEncerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]
        print("\nHistórico apagado!\n")
        continue

    if entrada.lower() == "historico":
        print(f"\nMensagens armazenadas: {len(historico)}\n")
        continue

    if not entrada:
        print("Digite uma mensagem válida.")
        continue

    historico.append({
        "role": "user",
        "content": entrada
    })

    try:
        texto = responder_chat(historico)

        historico.append({
            "role": "assistant",
            "content": texto
        })

        print(f"\n[Gemini] {texto}\n")

    except Exception as erro:
        historico.pop()
        print(f"\nErro ao consultar Gemini: {erro}\n")

Chat iniciado com Google Gemini!
Comandos: sair | limpar | historico

Você: Meu nome é Gustavo e estou trabalhando com um ESP32.

[Gemini] Olá, Gustavo! É um prazer ajudar você nos seus projetos com o ESP32. Por favor, descreva qual é o problema ou dúvida técnica que você está enfrentando no momento, como falhas de conexão Wi-Fi, leitura de sensores ou gravação de código. Estou à disposição para orientá-lo passo a passo na solução!

Você: Qual é o meu nome e qual dispositivo estou utilizando?

[Gemini] O seu nome é Gustavo e você está utilizando um microcontrolador ESP32. Como assistente técnico de IoT, estou pronto para ajudar com qualquer dúvida ou problema relacionado ao seu dispositivo. Basta me dizer qual é o seu próximo desafio no projeto!

Você: historico

Mensagens armazenadas: 5

Você: limpar

Histórico apagado!

Você: Qual é o meu nome e qual dispositivo estou utilizando?

[Gemini] Como sou um assistente de suporte técnico de IoT, não tenho acesso a informações pessoais como 

In [29]:
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é um assistente especializado em suporte técnico de IoT.  Sua função é ajud
[user] Qual é o meu nome e qual dispositivo estou utilizando?
[assistant] Como sou um assistente de suporte técnico de IoT, não tenho acesso a informações


**Observações do grupo (Etapa 3):**

- Ao trocar de modelo no meio da conversa, o novo modelo "lembrou" do que foi dito?
- Por que isso acontece (ou não acontece)?
- O que muda na forma como o histórico é enviado para o Gemini em relação ao Hugging Face?

(responda aqui)

---
## Etapa 4: Interface web com Gradio e controle de temperatura

Na atividade anterior a interface tinha o seletor de modelo.
Aqui o grupo adiciona um **slider de temperatura** para controlar a criatividade das respostas em tempo real.

In [ ]:
import gradio as gr

def texto_da_mensagem(conteudo):
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor, temperatura):
    if provedor == "Hugging Face":
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(
            messages=mensagens,
            max_tokens=300,
            temperature=temperatura,
        )
        return resposta.choices[0].message.content

    else:
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                max_output_tokens=300,
                temperature=temperatura,
            ),
        )
        return resposta.text

In [ ]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[
        gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo"),
        gr.Slider(minimum=0.0, maximum=1.0, step=0.1, value=0.7, label="Temperatura"),
    ],
    title="Assistente de Casa Inteligente",
    description="Pergunte sobre automação residencial, sensores e dispositivos conectados.",
    examples=[
        ["Como funciona uma lâmpada inteligente?", "Hugging Face", 0.7],
        ["Vale a pena usar tomadas conectadas?",   "Gemini",       0.3],
    ],
).launch(share=True)

**Observações do grupo (Etapa 4):**

- Tire um print da interface com o slider visível e coloque no README do repositório.
- Teste a mesma pergunta com temperatura 0.0 e 1.0 na interface. O que mudou?
- Qual combinação de modelo e temperatura funcionou melhor para o tema?

(responda aqui)

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5: API com sessões independentes

Na atividade anterior a API não guardava histórico entre chamadas.
Aqui cada `session_id` tem seu próprio histórico — duas sessões paralelas não compartilham contexto.

In [ ]:
%%writefile app_avaliacao.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# Token e system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

# Histórico por sessão
sessoes: dict[str, list] = {}

class Pergunta(BaseModel):
    session_id: str
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    if pergunta.session_id not in sessoes:
        sessoes[pergunta.session_id] = [{"role": "system", "content": SYSTEM_PROMPT}]

    historico = sessoes[pergunta.session_id]
    historico.append({"role": "user", "content": pergunta.mensagem})

    resposta = client.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    return {"session_id": pergunta.session_id, "resposta": texto}

@app.delete("/sessao/{session_id}")
def limpar_sessao(session_id: str):
    sessoes.pop(session_id, None)
    return {"status": "sessão apagada", "session_id": session_id}

In [ ]:
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"]      = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app_avaliacao:app", "--port", "8000"])
time.sleep(5)
print("Servidor rodando em http://localhost:8000")

In [ ]:
import requests

# Sessão A — apresenta o nome
requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Meu nome é Ana."})
r_a = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Qual é o meu nome?"})

# Sessão B — não apresentou o nome
r_b = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_b", "mensagem": "Qual é o meu nome?"})

print("Sessão A:", r_a.json()["resposta"])
print("Sessão B:", r_b.json()["resposta"])

In [ ]:
servidor.terminate()
print("Servidor encerrado.")

**Observações do grupo (Etapa 5 — Bônus):**

- A sessão A respondeu o nome corretamente? E a sessão B?
- O que mudou nessa API em relação à da atividade anterior?
- Cite um caso de uso real onde sessões independentes seriam necessárias.

(responda aqui)